# NVIDIA Cosmos 3 Nano — Deployment Tutorial on AMD ROCm

**Platform:** AMD Radeon Cloud (`vllm_omni:20260609`)  
**Model:** [`nvidia/Cosmos3-Nano`](https://huggingface.co/nvidia/Cosmos3-Nano)  
**Architecture:** Mixture-of-Transformers (MoT) — 16B total  
**License:** [OpenMDW 1.1](https://openmdw.ai/license/1-1/)

Cosmos 3 Nano is an omnimodal world foundation model that generates video, image, audio, and robot action outputs from text, image, video, and action trajectory inputs. It is designed for Physical AI applications: robotics, autonomous driving, and industrial simulation.

---

## Two Deployment Paths

| | Path A: Diffusers (this notebook) | Path B: vLLM-Omni Server |
|---|---|---|
| Best for | Tutorial / research / prototyping | Production API serving |
| Code style | In-process Python | HTTP client → server |
| VRAM footprint | Loaded into Python process | Managed by vLLM runtime |
| ROCm support | ✅ via PyTorch HIP backend | ✅ native in `vllm_omni` image |

This notebook covers **Path A (Diffusers)** for interactive exploration, then demonstrates **Path B (vLLM-Omni server)** for production deployment.

---

## Hardware Requirements

| Model | Parameters | VRAM (BF16) | Recommended AMD GPU |
|---|---|---|---|
| **Cosmos3-Nano** | **16B** | **~32 GB** | **1× MI250X or 1× MI300X** |
| Cosmos3-Super | 64B | ~128 GB | 1× MI300X (192GB) or 4× MI250X |

> **Note:** Only BF16 precision is officially tested. FP16/FP8/FP4 are not supported.

## 0. Environment Verification

Verify that the ROCm stack is active and GPUs are visible. On AMD hardware, PyTorch exposes the HIP backend through the standard `torch.cuda` API — `torch.cuda.is_available()` returns `True` and `torch.version.hip` contains the ROCm version.

In [ ]:
import subprocess, sys, os

# --- ROCm / GPU info ---
print("=" * 55)
print("ROCm & GPU Environment")
print("=" * 55)

result = subprocess.run(["rocm-smi", "--showproductname", "--showmeminfo", "vram"],
                        capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "rocm-smi not found — skipping")

import torch
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA (HIP) avail : {torch.cuda.is_available()}")
print(f"ROCm/HIP version : {getattr(torch.version, 'hip', 'N/A')}")
print(f"GPU count        : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / (1024**3)
    print(f"  GPU {i}: {props.name}  |  VRAM: {vram_gb:.1f} GB")

total_vram = sum(
    torch.cuda.get_device_properties(i).total_memory
    for i in range(torch.cuda.device_count())
) / (1024**3)
print(f"\nTotal VRAM across all GPUs: {total_vram:.1f} GB")

COSMOS_NANO_REQ_GB = 32
if total_vram >= COSMOS_NANO_REQ_GB:
    print(f"✅ Sufficient VRAM for Cosmos3-Nano ({COSMOS_NANO_REQ_GB} GB required)")
else:
    print(f"⚠️  Total VRAM ({total_vram:.0f} GB) may be insufficient for Cosmos3-Nano.")
    print("   Cosmos3-Nano requires ~32 GB VRAM (BF16).")

## 1. Install Dependencies

The `vllm_omni` container ships with PyTorch ROCm and vLLM-Omni. We additionally need:
- **`diffusers` from git** — `Cosmos3OmniPipeline` is not on PyPI yet (as of June 2026)
- **`cosmos_guardrail`** — NVIDIA's content safety filter (optional but recommended)
- **`huggingface_hub`** — for downloading model assets and example prompts

In [ ]:
import subprocess, sys

def pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Install warning: {result.stderr[-500:]}")
    return result.returncode == 0

print("Installing diffusers from source (required for Cosmos3OmniPipeline)...")
pip("git+https://github.com/huggingface/diffusers.git")

print("Installing supporting libraries...")
pip("accelerate", "transformers", "sentencepiece")
pip("huggingface_hub[cli]")
pip("imageio", "imageio-ffmpeg", "av", "Pillow")
pip("requests")

# Optional: NVIDIA safety guardrails
try:
    pip("cosmos_guardrail")
    GUARDRAILS = True
    print("cosmos_guardrail installed — safety checks enabled.")
except Exception:
    GUARDRAILS = False
    print("cosmos_guardrail not available — proceeding without safety filter.")

print("\nDone. If this is the first run, restart the kernel now to pick up the new diffusers install.")

## 2. Hugging Face Authentication

Cosmos3-Super requires accepting the [OpenMDW 1.1 license](https://openmdw.ai/license/1-1/) on the Hugging Face model page before downloading weights.  
Set your token either via the environment variable `HF_TOKEN` or run `huggingface-cli login` in a terminal.

In [ ]:
import os
from huggingface_hub import login, whoami

HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    user = whoami()
    print(f"Logged in as: {user['name']}")
else:
    print("HF_TOKEN environment variable not set.")
    print("Either set HF_TOKEN or uncomment the line below and paste your token:")
    # login()  # interactive prompt

MODEL_ID = "nvidia/Cosmos3-Nano"
print(f"\nTarget model: {MODEL_ID}")

## 3. ROCm Environment Variables

A few environment variables improve performance and compatibility on AMD GPUs.

In [ ]:
import os

# Enable AMD's Triton-based Flash Attention kernel (MI200/MI300 CDNA GPUs)
os.environ["FLASH_ATTENTION_TRITON_AMD_ENABLE"] = "TRUE"

# Use better error messages for HIP kernel failures during debugging
os.environ["TORCH_USE_HIP_DSA"] = "1"

# Ensure BF16 is used (the only tested precision for Cosmos3)
# FP16 can overflow on long video generation; FP32 doubles memory usage
os.environ["PYTORCH_NO_CUDA_MEMORY_CACHING"] = "0"  # keep caching on for performance

# Optional: restrict to specific GPUs (equivalent of CUDA_VISIBLE_DEVICES on ROCm)
# os.environ["ROCR_VISIBLE_DEVICES"] = "0,1,2,3"

import torch
# Verify BF16 support on this GPU
if torch.cuda.is_available():
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"BF16 supported on device 0: {bf16_ok}")
    if not bf16_ok:
        print("WARNING: BF16 not supported. Cosmos3 may not run correctly on this GPU.")

print("ROCm environment configured.")

---

## Path A — Diffusers (Direct In-Process Inference)

### 4. Load the Pipeline

**Key points vs. the generic `DiffusionPipeline` pattern:**
- Use `Cosmos3OmniPipeline`, not `DiffusionPipeline`
- The kwarg is `torch_dtype=`, not `dtype=`
- Output is always `.video`, never `.images` — even for single-frame (image) generation
- `device_map="cuda"` works on AMD because ROCm exposes AMD GPUs through the CUDA API surface (HIP)

Multi-GPU (`device_map="auto"`) distributes the 64B model across all visible GPUs automatically via Accelerate's device map.

In [ ]:
import torch
from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler

n_gpus = torch.cuda.device_count()
device_map = "auto"
print(f"Loading model across {n_gpus} GPU(s) with device_map='{device_map}'")

pipe = Cosmos3OmniPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,   # only BF16 is tested; do NOT use float16 or float32
    device_map=device_map,
    enable_safety_checker=GUARDRAILS,
)

# Cosmos3 uses UniPC multistep scheduler with flow_shift=10.0
pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config, flow_shift=10.0
)

print("Pipeline loaded successfully.")

### 5a. Text-to-Image (Single Frame)

Setting `num_frames=1` generates a single high-resolution image. The output is still returned as `.video[0]`, which is a PIL Image object for single frames.

In [ ]:
import json
from IPython.display import display

prompt_t2i = (
    "A medium shot of a modern robotics research laboratory with white walls and a gray floor. "
    "A 6-DOF robotic arm with a metallic finish is mounted on a clean white workbench, its gripper "
    "positioned above a row of small colored objects arranged in a line. Overhead fluorescent lighting "
    "casts clean even shadows. A large monitor on the wall displays a trajectory planning software "
    "interface. The scene is photorealistic, high detail, 4K quality."
)

result = pipe(
    prompt=prompt_t2i,
    num_frames=1,        # 1 = single image
    height=720,
    width=1280,
    num_inference_steps=35,
    guidance_scale=6.0,
    generator=torch.Generator(device="cuda").manual_seed(42),
)

img = result.video[0]
img.save("/tmp/cosmos3_nano_t2i.jpg", format="JPEG", quality=90)
print("Saved: /tmp/cosmos3_nano_t2i.jpg")
display(img)

### 5b. Text-to-Video

`num_frames=189` at 24fps produces a ~7.875s clip — the default generation length in the model card. Lower `num_inference_steps` (e.g. 20) reduces generation time at a small quality cost.

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video

prompt_t2v = (
    "The video begins with a close-up view of a robotic arm operating in an industrial warehouse. "
    "The robot's gripper smoothly picks up a red cylindrical object from a conveyor belt and places "
    "it precisely into a designated bin. The motion is fluid, with no hesitation. The background "
    "shows organized shelving units under bright LED lights. Camera is static, observing the full "
    "pick-and-place cycle from a slight overhead angle. Photorealistic, 4K quality."
)

result_t2v = pipe(
    prompt=prompt_t2v,
    num_frames=189,          # ~7.875s @ 24fps
    height=720,
    width=1280,
    num_inference_steps=35,  # reduce to 20 for faster inference
    guidance_scale=6.0,
    generator=torch.Generator(device="cuda").manual_seed(17),
)

out_path = "/tmp/cosmos3_nano_t2v.mp4"
export_to_video(result_t2v.video, out_path, fps=24)
print(f"Saved: {out_path}")
Video(out_path, width=640)

### 5c. Image-to-Video

Provide a conditioning image (PIL Image or local path) alongside the text prompt. The model generates a video that continues the visual story from the given frame.

In [ ]:
from PIL import Image
import requests
from io import BytesIO
from diffusers.utils import export_to_video
from IPython.display import Video, display

img_url = "https://huggingface.co/nvidia/Cosmos3-Nano/resolve/main/assets/example_i2v_input.jpg"
response = requests.get(img_url, timeout=30)
cond_image = Image.open(BytesIO(response.content)).convert("RGB")
print(f"Conditioning image size: {cond_image.size}")
display(cond_image)

prompt_i2v = (
    "Starting from this frame, a robotic arm begins to move with precision across the workspace. "
    "The gripper extends toward the nearest object, grasps it gently, and lifts it in a smooth arc. "
    "The background remains static. Photorealistic, steady camera, 4K quality."
)

result_i2v = pipe(
    prompt=prompt_i2v,
    image=cond_image,    # conditioning frame; triggers image-to-video mode
    num_frames=189,
    height=720,
    width=1280,
    num_inference_steps=35,
    guidance_scale=6.0,
    generator=torch.Generator(device="cuda").manual_seed(17),
)

out_path_i2v = "/tmp/cosmos3_nano_i2v.mp4"
export_to_video(result_i2v.video, out_path_i2v, fps=24)
print(f"Saved: {out_path_i2v}")
Video(out_path_i2v, width=640)

---

## Path B — vLLM-Omni Server (Production API)

This path uses the vLLM-Omni server already present in the `vllm_omni:20260609` container. It exposes an OpenAI-compatible REST API, making it suitable for multi-user serving and integration with existing inference infrastructure.

### 6. Start the vLLM-Omni Server

**ROCm-specific differences from the NVIDIA model card:**
- `--use-hsdp` and `--ulysses-degree` are NVLink-specific flags — omit them
- Use `--tensor-parallel-size` for multi-GPU distribution
- `HIP_VISIBLE_DEVICES` (or `ROCR_VISIBLE_DEVICES`) instead of `CUDA_VISIBLE_DEVICES`

In [ ]:
import subprocess, time, requests as req, os, torch

n_gpus = torch.cuda.device_count()
SERVER_PORT = 8000
SERVER_URL = f"http://localhost:{SERVER_PORT}"

# NOTE: If you already ran Path A, free GPU memory first:
# del pipe; import gc; gc.collect(); torch.cuda.empty_cache()

serve_cmd = [
    "vllm", "serve", MODEL_ID,
    "--omni",
    "--host", "0.0.0.0",
    "--port", str(SERVER_PORT),
    "--tensor-parallel-size", str(n_gpus),  # distribute across all GPUs
    "--dtype", "bfloat16",
    "--init-timeout", "1800",  # allow 30 min for 64B model load
]

print("Starting vLLM-Omni server...")
print("Command:", " ".join(serve_cmd))
print(f"Tensor parallel size: {n_gpus} GPU(s)")

server_proc = subprocess.Popen(
    serve_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Poll until server is ready (health endpoint becomes 200)
print("Waiting for server to be ready (may take 5-15 min for first load)...")
for attempt in range(120):
    time.sleep(10)
    try:
        r = req.get(f"{SERVER_URL}/health", timeout=5)
        if r.status_code == 200:
            print(f"\nServer ready after {(attempt+1)*10}s")
            break
    except Exception:
        pass
    if attempt % 3 == 0:
        # Show last line of server output
        line = server_proc.stdout.readline().strip()
        if line:
            print(f"  Server log: {line}")
else:
    print("Server did not become ready within 20 minutes. Check logs above.")

### 7. Text-to-Video via REST API

In [ ]:
import requests, json
from pathlib import Path
from IPython.display import Video

prompt_api = (
    "An autonomous vehicle navigates a rain-slicked urban intersection at night. "
    "Streetlights reflect off wet asphalt. Pedestrians with umbrellas cross at a crosswalk. "
    "The vehicle smoothly decelerates, yields, then accelerates through the intersection. "
    "Dashcam perspective, realistic, 4K."
)

negative_prompt = {
    "text": "blurry, low quality, artifact, distorted, flickering, watermark, text overlay"
}

data = {
    "prompt": prompt_api,
    "negative_prompt": json.dumps(negative_prompt),
    "size": "1280x720",
    "num_frames": "189",
    "fps": "24",
    "num_inference_steps": "35",
    "guidance_scale": "6.0",
    "flow_shift": "10.0",
    "seed": "42",
}

print("Sending text-to-video request...")
response = requests.post(
    f"{SERVER_URL}/v1/videos/sync",
    data=data,
    headers={"Accept": "video/mp4"},
    timeout=300,
)
response.raise_for_status()

out_path = Path("/tmp/cosmos3_nano_api_t2v.mp4")
out_path.write_bytes(response.content)
print(f"Saved: {out_path}  ({len(response.content) / 1024:.0f} KB)")
Video(str(out_path), width=640)

### 8. Image-to-Video via REST API

In [ ]:
import requests, json, mimetypes
from pathlib import Path
from io import BytesIO
from PIL import Image
from IPython.display import Video, display

img_url = "https://huggingface.co/nvidia/Cosmos3-Nano/resolve/main/assets/example_i2v_input.jpg"
img_bytes = requests.get(img_url, timeout=30).content
image_path = Path("/tmp/example_i2v_input.jpg")
image_path.write_bytes(img_bytes)
display(Image.open(image_path))

data = {
    "prompt": "The scene continues as the robotic arm completes its pick operation and pivots to place the object.",
    "size": "1280x720",
    "num_frames": "189",
    "fps": "24",
    "num_inference_steps": "35",
    "guidance_scale": "6.0",
    "flow_shift": "10.0",
    "seed": "17",
}

print("Sending image-to-video request...")
with image_path.open("rb") as f:
    response = requests.post(
        f"{SERVER_URL}/v1/videos/sync",
        data=data,
        files={"input_reference": (image_path.name, f, "image/jpeg")},
        headers={"Accept": "video/mp4"},
        timeout=300,
    )
response.raise_for_status()

out_path = Path("/tmp/cosmos3_nano_api_i2v.mp4")
out_path.write_bytes(response.content)
print(f"Saved: {out_path}  ({len(response.content) / 1024:.0f} KB)")
Video(str(out_path), width=640)

### 9. Shut Down the Server

In [ ]:
if 'server_proc' in dir() and server_proc.poll() is None:
    server_proc.terminate()
    server_proc.wait(timeout=10)
    print("vLLM-Omni server terminated.")
else:
    print("Server already stopped or was not started in this session.")

---

## Tips & Troubleshooting on AMD ROCm

| Issue | Fix |
|---|---|
| `RuntimeError: HIP error: no kernel image is available` | GPU architecture not in the compiled wheel. Set `HSA_OVERRIDE_GFX_VERSION=11.0.0` (RDNA3) or `9.4.0` (MI300X) |
| OOM during 16B model load | Use `device_map="auto"` to spread across GPUs, or reduce resolution/num_frames |
| BF16 errors on older GPUs | Only CDNA2+ (MI210/MI250/MI300) and RDNA3+ support native BF16; older cards will be slow or error |
| Flash Attention not kicking in | Confirm `FLASH_ATTENTION_TRITON_AMD_ENABLE=TRUE` and that `aotriton` is installed in the container |
| vLLM-Omni server crashes on startup | Try `--enforce-eager` to disable CUDA graph compilation, which can fail on some ROCm configurations |
| Slow first inference on Diffusers path | Normal — PyTorch ROCm compiles Triton kernels on first run; subsequent calls are faster |

## References

- [NVIDIA Cosmos 3 Nano Model Card (HuggingFace)](https://huggingface.co/nvidia/Cosmos3-Nano)
- [Cosmos 3 GitHub](https://github.com/nvidia/cosmos)
- [Cosmos 3 Technical Report](https://research.nvidia.com/labs/cosmos-lab/cosmos3/technical-report.pdf)
- [Diffusers Cosmos3 Documentation](https://huggingface.co/docs/diffusers/main/en/api/pipelines/cosmos3)
- [ROCm Documentation](https://rocm.docs.amd.com)
- [learn-world-model Project](https://github.com/qiwang067/learn-world-model)
